# This file will find the closest train station for each properties by using Open Route Service.

## 1. Using PTV shape file to find the stations' central coordinates

In [1]:
# import needed libraries
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely import wkt
from shapely.geometry import Point
from geopy.distance import great_circle

In [2]:
# sf stands for shape file 
# read files and select needed features
PTV_sf = gpd.read_file("../data/landing/external_data/shapefile/PTV_Train_Station/ll_gda94/esrishape/whole_of_dataset/victoria/PTV/PTV_TRAIN_STATION_PLATFORM.shp")
# Select the 'geometry' and 'STATION' columns, and rename 'STATION' to lowercase
PTV_sf = PTV_sf[['geometry', 'STATION']]
PTV_sf.rename(columns={'STATION': 'station_name'}, inplace=True)
PTV_sf

,geometry,station_name
0,"POLYGON ((142.03168 -36.45305, 142.03165 -36.4...",Dimboola
1,"POLYGON ((143.92485 -35.73359, 143.92440 -35.7...",Kerang
2,"POLYGON ((143.92409 -35.73242, 143.92452 -35.7...",Kerang
3,"POLYGON ((144.75318 -36.13053, 144.75322 -36.1...",Echuca
4,"POLYGON ((144.69826 -36.36258, 144.69876 -36.3...",Rochester
...,...,...
701,"POLYGON ((145.08915 -37.87483, 145.08914 -37.8...",Holmesglen
702,"POLYGON ((144.36046 -38.15948, 144.36042 -38.1...",South Geelong
703,"POLYGON ((144.36872 -38.17557, 144.36813 -38.1...",Breakwater
704,"POLYGON ((144.30591 -38.21588, 144.30788 -38.2...",NaN


In [3]:
# pre-process to the shape file: 
# 1. delete station with NaN (invalid station)
PTV_sf = PTV_sf.dropna(subset=['station_name'])

# 2. Calculate the center point of each site and store it in a new column 'station centroid'
# UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.
PTV_sf['station_centroid'] = PTV_sf['geometry'].centroid
PTV_sf

/tmp/ipykernel_5145/72965145.py:7: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  PTV_sf['station_centroid'] = PTV_sf['geometry'].centroid
/home/ximing/.local/lib/python3.10/site-packages/geopandas/geodataframe.py:1538: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


,geometry,station_name,station_centroid
0,"POLYGON ((142.03168 -36.45305, 142.03165 -36.4...",Dimboola,POINT (142.03098 -36.45202)
1,"POLYGON ((143.92485 -35.73359, 143.92440 -35.7...",Kerang,POINT (143.92465 -35.73311)
2,"POLYGON ((143.92409 -35.73242, 143.92452 -35.7...",Kerang,POINT (143.92428 -35.73288)
3,"POLYGON ((144.75318 -36.13053, 144.75322 -36.1...",Echuca,POINT (144.75329 -36.13104)
4,"POLYGON ((144.69826 -36.36258, 144.69876 -36.3...",Rochester,POINT (144.69855 -36.36220)
...,...,...,...
699,"POLYGON ((145.12728 -37.87531, 145.12897 -37.8...",Mount Waverley,POINT (145.12814 -37.87532)
700,"POLYGON ((145.12721 -37.87523, 145.12899 -37.8...",Mount Waverley,POINT (145.12812 -37.87524)
701,"POLYGON ((145.08915 -37.87483, 145.08914 -37.8...",Holmesglen,POINT (145.09001 -37.87461)
702,"POLYGON ((144.36046 -38.15948, 144.36042 -38.1...",South Geelong,POINT (144.35965 -38.15893)


In [4]:
# save as the CSV file fot further usage
PTV_sf.to_csv("../data/raw/external_data/PTV_station_data.csv", index=False)
# read the csv and select features 'station_name' and 'station_centroid'
PTV_station_data = pd.read_csv("../data/raw/external_data/PTV_station_data.csv")
PTV_station_data = PTV_station_data[['station_name', 'station_centroid']]
PTV_station_data

,station_name,station_centroid
0,Dimboola,POINT (142.03097879572368 -36.452019916533985)
1,Kerang,POINT (143.92465096653459 -35.73311015757247)
2,Kerang,POINT (143.92427795758204 -35.73288167764773)
3,Echuca,POINT (144.75328509216718 -36.13103835849756)
4,Rochester,POINT (144.69855282297706 -36.36220456920009)
...,...,...
680,Mount Waverley,POINT (145.1281391923223 -37.87532263838918)
681,Mount Waverley,POINT (145.1281158277881 -37.87523548389401)
682,Holmesglen,POINT (145.09001211543153 -37.87460973544187)
683,South Geelong,POINT (144.35965242241852 -38.15893436083373)


In [5]:
# Extract longitude and latitude and store them as new column 'coordinates',
# format the coordinates in the 'coordinates' column into a string in the format '[latitude, longitude]'
PTV_station_data['station_centroid'] = PTV_station_data['station_centroid'].apply(wkt.loads)
PTV_station_data['station_coordinates'] = PTV_station_data['station_centroid'].apply(lambda geom: [geom.y, geom.x])
PTV_station_data['station_coordinates'] = PTV_station_data['station_coordinates'].apply(lambda coords: f"[{coords[0]}, {coords[1]}]")
PTV_station_data

,station_name,station_centroid,station_coordinates
0,Dimboola,POINT (142.03097879572368 -36.452019916533985),"[-36.452019916533985, 142.03097879572368]"
1,Kerang,POINT (143.92465096653459 -35.73311015757247),"[-35.73311015757247, 143.92465096653459]"
2,Kerang,POINT (143.92427795758204 -35.73288167764773),"[-35.73288167764773, 143.92427795758204]"
3,Echuca,POINT (144.75328509216718 -36.13103835849756),"[-36.13103835849756, 144.75328509216718]"
4,Rochester,POINT (144.69855282297706 -36.36220456920009),"[-36.36220456920009, 144.69855282297706]"
...,...,...,...
680,Mount Waverley,POINT (145.1281391923223 -37.87532263838918),"[-37.87532263838918, 145.1281391923223]"
681,Mount Waverley,POINT (145.1281158277881 -37.87523548389401),"[-37.87523548389401, 145.1281158277881]"
682,Holmesglen,POINT (145.09001211543153 -37.87460973544187),"[-37.87460973544187, 145.09001211543153]"
683,South Geelong,POINT (144.35965242241852 -38.15893436083373),"[-38.15893436083373, 144.35965242241852]"


In [6]:
# we find some stations have many platforms, but we only need one as the coordinates to claculate the distance
station_counts = PTV_station_data['station_name'].value_counts()
print(station_counts)

station_name
Southern Cross     20
Flinders St        14
Richmond           10
North Melbourne     6
Footscray           6
                   ..
Murchison East      1
Mooroopna           1
Shepparton          1
Showgrounds         1
Breakwater          1
Name: count, Length: 331, dtype: int64


In [7]:
# Keep a unique record of each station_name
PTV_station_data_unique = PTV_station_data.drop_duplicates(subset=['station_name'])
# save as the CSV file fot further usage
PTV_station_data_unique = PTV_station_data_unique[['station_name', 'station_coordinates']]
PTV_station_data_unique.to_csv("../data/raw/external_data/PTV_station_data.csv", index=False)
PTV_station_data_unique

,station_name,station_coordinates
0,Dimboola,"[-36.452019916533985, 142.03097879572368]"
1,Kerang,"[-35.73311015757247, 143.92465096653459]"
3,Echuca,"[-36.13103835849756, 144.75328509216718]"
4,Rochester,"[-36.36220456920009, 144.69855282297706]"
5,Elmore,"[-36.49489611863695, 144.60763619157052]"
...,...,...
676,Gardiner,"[-37.852867997222994, 145.0508099029106]"
678,Jordanville,"[-37.873708994762815, 145.11246954380377]"
680,Mount Waverley,"[-37.87532263838918, 145.1281391923223]"
683,South Geelong,"[-38.15893436083373, 144.35965242241852]"


## 2. Find the top 3 closest station for each properties

In [8]:
# read property data and station data
property_data = pd.read_csv("../data/curated/merged_data/merged_data_with_facility.csv")
station_data = pd.read_csv("../data/raw/external_data/PTV_station_data.csv")
station_data = station_data[['station_name', 'station_coordinates']]
station_data

,station_name,station_coordinates
0,Dimboola,"[-36.452019916533985, 142.03097879572368]"
1,Kerang,"[-35.73311015757247, 143.92465096653459]"
2,Echuca,"[-36.13103835849756, 144.75328509216718]"
3,Rochester,"[-36.36220456920009, 144.69855282297706]"
4,Elmore,"[-36.49489611863695, 144.60763619157052]"
...,...,...
326,Gardiner,"[-37.852867997222994, 145.0508099029106]"
327,Jordanville,"[-37.873708994762815, 145.11246954380377]"
328,Mount Waverley,"[-37.87532263838918, 145.1281391923223]"
329,South Geelong,"[-38.15893436083373, 144.35965242241852]"


In [9]:
#The function accepts a string containing coordinates in the format "[latitude, longitude]", 
# then parses it and returns a tuple of floats containing the latitude and longitude
def parse_coordinates(coord_str):
    parts = coord_str.strip('[]').split(',')
    return (float(parts[0]), float(parts[1]))

# parse coordinate strings for property and station data
property_coordinates = property_data['coordinates'].apply(parse_coordinates)
station_coordinates = station_data['station_coordinates'].apply(parse_coordinates)
# This function is used to find the geometric information of the three closest stations to a given property coordinate. 
def find_nearest_station(property_coord):
    if np.isnan(property_coord).any():
        return [None, None, None]  # If the property coordinates contain NaN values, return three None
    distances = [
        great_circle(property_coord, station_coord).kilometers
        if not np.isnan(station_coord).any() else float('inf')
        for station_coord in station_coordinates
    ]
    # find the index of the two closest stations
    closest_indices = np.argsort(distances)[:2]
    # get geometry information for the three closest stations
    closest_station = [station_data.loc[i, 'station_coordinates'] for i in closest_indices]
    return closest_station

# find the three nearest stations for each property
property_data['closest_station'] = property_coordinates.apply(find_nearest_station)
property_data






,name,rental_price,num_bedroom,num_bathroom,num_parking,postcode,coordinates,property_geometry,sa2_code,sa2_name,...,closest_school,school_distance(KM),closest_hospital,hospital_distance(KM),closest_mall,mall_distance(KM),closest_park,park_distance(KM),closest_station,station_distance(KM)
0,31 Chittagong Drive Clyde North VIC 3978,575.0,4,2,2.0,3978.0,"[-38.1053122, 145.3570863]",POINT (145.3570863 -38.1053122),212031556.0,Clyde North - South,...,"[-38.10602, 145.37876]",1.898006,"[-38.045325, 145.347181]",6.726397,"[-38.1184718, 145.3213262]",3.453903,"[-38.13958090963712, 145.36252669303045]",3.840116,"[[-38.05104856543076, 145.3665303282505], [-38...",6.090210
1,50 Elmtree Crescent Clyde North VIC 3978,560.0,4,2,2.0,3978.0,"[-38.0825712, 145.3561984]",POINT (145.3561984 -38.0825712),212031555.0,Clyde North - North,...,"[-38.08468, 145.3638]",0.705427,"[-38.045325, 145.347181]",4.216162,"[-38.0604189, 145.3394612]",2.866025,"[-38.03401014838179, 145.37166338662394]",5.566924,"[[-38.05104856543076, 145.3665303282505], [-38...",3.619981
2,7 Mortdale Lane Clyde North VIC 3978,490.0,2,2,1.0,3978.0,"[-38.0961758, 145.3800644]",POINT (145.3800644 -38.0961758),212031556.0,Clyde North - South,...,"[-38.10602, 145.37876]",1.100561,"[-38.045325, 145.347181]",6.344909,"[-38.0604189, 145.3394612]",5.332842,"[-38.13958090963712, 145.36252669303045]",5.064419,"[[-38.0663192834778, 145.4116572812351], [-38....",4.320648
3,54 Walhallow Drive Clyde North VIC 3978,540.0,4,2,1.0,3978.0,"[-38.1133324, 145.3457396]",POINT (145.3457396 -38.1133324),212031556.0,Clyde North - South,...,"[-38.11488, 145.33828]",0.674921,"[-38.113312, 145.280832]",5.678594,"[-38.1184718, 145.3213262]",2.210922,"[-38.13958090963712, 145.36252669303045]",3.267265,"[[-38.0995872148084, 145.2804738077291], [-38....",5.911467
4,10 Sicily Road Clyde North VIC 3978,520.0,4,2,2.0,3978.0,"[-38.1295789, 145.3642993]",POINT (145.3642993 -38.1295789),212031303.0,Cranbourne South,...,"[-38.12955, 145.33886]",2.225124,"[-38.113312, 145.280832]",7.522231,"[-38.1184718, 145.3213262]",3.956745,"[-38.13958090963712, 145.36252669303045]",1.122928,"[[-38.0995872148084, 145.2804738077291], [-38....",8.056215
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8797,61 Tongue Street Yarraville VIC 3013,630.0,2,1,0.0,3013.0,"[-37.8131463, 144.8909053]",POINT (144.8909053 -37.8131463),213031352.0,Yarraville,...,"[-37.8137, 144.8899]",0.107655,"[-37.800508, 144.895155]",1.454065,"[-37.8016476, 144.8977057]",1.411290,"[-37.83084513979606, 144.91345002848863]",2.791844,"[[-37.81567190405462, 144.89006145414797], [-3...",0.290453
8798,47 Mill Avenue Yarraville VIC 3013,730.0,4,3,2.0,3013.0,"[-37.8222822, 144.872198]",POINT (144.872198 -37.8222822),213031352.0,Yarraville,...,"[-37.82104, 144.87443]",0.239821,"[-37.797407, 144.887421]",3.072331,"[-37.828989, 144.84627]",2.396280,"[-37.83084513979606, 144.91345002848863]",3.746179,"[[-37.83072411396968, 144.88584876958876], [-3...",1.522703
8799,12 Adeney Street Yarraville VIC 3013,450.0,3,1,2.0,3013.0,"[-37.8163817, 144.8666543]",POINT (144.8666543 -37.8163817),213031352.0,Yarraville,...,"[-37.8126, 144.87466]",0.819385,"[-37.797407, 144.887421]",2.789294,"[-37.828989, 144.84627]",2.273966,"[-37.83084513979606, 144.91345002848863]",4.413664,"[[-37.7993395010312, 144.86344949623717], [-37...",1.915810
8800,229B Somerville Road Yarraville VIC 3013,300.0,1,1,0.0,3013.0,"[-37.8124289, 144.8779569]",POINT (144.8779569 -37.8124289),213031352.0,Yarraville,...,"[-37.8126, 144.87466]",0.290245,"[-37.797407, 144.887421]",1.865866,"[-37.8016476, 144.8977057]",2.108881,"[-37.83084513979606, 144.91345002848863]",3.729966,"[[-37.81567190405462, 144.89006145414797], [-3...",1.122803


In [10]:
property_data_1800_2700= property_data.iloc[1800:2700]
property_data_2700_3600= property_data.iloc[2700:3600]
property_data_3600_4500= property_data.iloc[3600:4500]
property_data_4500_5400= property_data.iloc[4500:5400]
property_data_5400_6300= property_data.iloc[5400:6300]
property_data_6300_7200= property_data.iloc[6300:7200]
property_data_7200_8100= property_data.iloc[7200:8100]
property_data_8100_8802= property_data.iloc[8100:8802]


In [11]:
sample_property_data =  property_data.sample(frac=0.001, random_state=42).reset_index()
sample_property_data

,index,name,rental_price,num_bedroom,num_bathroom,num_parking,postcode,coordinates,property_geometry,sa2_code,...,closest_school,school_distance(KM),closest_hospital,hospital_distance(KM),closest_mall,mall_distance(KM),closest_park,park_distance(KM),closest_station,station_distance(KM)
0,95,205/3 Duggan Street Brunswick West VIC 3055,500.0,2,2,1.0,3055.0,"[-37.7590499, 144.9401487]",POINT (144.9401487 -37.7590499),206011107.0,...,"[-37.75671, 144.94359]",0.399025,"[-37.754369, 144.958398]",1.686666,"[-37.765648, 144.922669]",1.702744,"[-37.79769541477594, 144.9686729570712]",4.974984,"[[-37.7613663414288, 144.96070676123415], [-37...",1.825495
1,5556,9/45-47 Hotham Street St Kilda East VIC 3183,460.0,2,1,1.0,3183.0,"[-37.8643908, 145.0001134]",POINT (145.0001134 -37.8643908),206051134.0,...,"[-37.86812, 145.00034]",0.415146,"[-37.869799, 145.003096]",0.655887,"[-37.868967, 144.980617]",1.785478,"[-37.81559014496639, 145.01414518728387]",5.564532,"[[-37.869599163825, 144.99354432710513], [-37....",0.817268
2,4685,2/50 Coveside Avenue Safety Beach VIC 3936,570.0,3,2,2.0,3936.0,"[-38.3180777, 144.9986046]",POINT (144.9986046 -38.3180777),214021377.0,...,"[-38.34184, 145.01011]",2.826424,"[-38.230174, 145.041697]",10.473329,"[-38.333538, 144.964645]",3.425052,"[-38.295596175105345, 145.0009026929488]",2.507864,"[[-38.337153505967834, 145.17807450383012], [-...",15.798210
3,2815,302/108 Munster Terrace North Melbourne VIC 3051,420.0,1,1,0.0,3051.0,"[-37.8006367, 144.943377]",POINT (144.943377 -37.8006367),206041506.0,...,"[-37.79873, 144.95064]",0.672438,"[-37.794997, 144.950859]",0.908535,"[-37.807962, 144.955688]",1.354004,"[-37.79769541477594, 144.9686729570712]",2.246497,"[[-37.80715648260668, 144.94211411121677], [-3...",0.733409
4,360,7 Erinvale Close Mooroolbark VIC 3138,485.0,2,2,1.0,3138.0,"[-37.7895549, 145.311328]",POINT (145.311328 -37.7895549),211051281.0,...,"[-37.78934, 145.31618]",0.427033,"[-37.756319, 145.351204]",5.093318,"[-37.799301795285, 145.28299318557]",2.715393,"[-37.7656957477457, 145.28367776485888]",3.597779,"[[-37.7848042857515, 145.31262566871925], [-37...",0.540413
5,5571,2/20 Leonard Street Ringwood VIC 3134,560.0,2,2,1.0,3134.0,"[-37.8051011, 145.240962]",POINT (145.240962 -37.8051011),211031266.0,...,"[-37.79975, 145.2376]",0.664299,"[-37.811523, 145.243629]",0.751540,"[-37.812672, 145.229157]",1.335754,"[-37.7656957477457, 145.28367776485888]",5.769766,"[[-37.81165061732653, 145.25085625363403], [-3...",1.133990
6,4001,2C Hill Street Thornbury VIC 3071,530.0,2,1,1.0,3071.0,"[-37.7546888, 144.9845667]",POINT (144.9845667 -37.7546888),206021112.0,...,"[-37.7477, 144.99061]",0.941393,"[-37.754369, 144.958398]",2.300907,"[-37.751407373871, 145.00238980791]",1.608879,"[-37.79769541477594, 144.9686729570712]",4.981970,"[[-37.75501379889045, 144.99862874026115], [-3...",1.236793
7,6822,13 Hafey Court Eureka VIC 3350,370.0,3,2,1.0,3350.0,"[-37.5669554, 143.8835074]",POINT (143.8835074 -37.5669554),201011481.0,...,"[-37.57297, 143.87591]",0.946381,"[-37.559658, 143.847041]",3.315069,"[-37.53431, 143.823885]",6.387798,"[-37.544825235958534, 143.8746539612468]",2.581560,"[[-37.558730747496156, 143.85995735127779], [-...",2.268300
8,3404,1501/39 Lonsdale Street Melbourne VIC 3000,500.0,2,1,0.0,3000.0,"[-37.8099303, 144.9711262]",POINT (144.9711262 -37.8099303),206041503.0,...,"[-37.8103, 144.97227]",0.108566,"[-37.808521, 144.97476]",0.355620,"[-37.811333, 144.969441]",0.215045,"[-37.81879933234266, 144.96928967418705]",0.999301,"[[-37.81099289241648, 144.97308849041195], [-3...",0.208991


In [12]:
sample_property_data =  property_data.head(2)
sample_property_data

,name,rental_price,num_bedroom,num_bathroom,num_parking,postcode,coordinates,property_geometry,sa2_code,sa2_name,...,closest_school,school_distance(KM),closest_hospital,hospital_distance(KM),closest_mall,mall_distance(KM),closest_park,park_distance(KM),closest_station,station_distance(KM)
0,31 Chittagong Drive Clyde North VIC 3978,575.0,4,2,2.0,3978.0,"[-38.1053122, 145.3570863]",POINT (145.3570863 -38.1053122),212031556.0,Clyde North - South,...,"[-38.10602, 145.37876]",1.898006,"[-38.045325, 145.347181]",6.726397,"[-38.1184718, 145.3213262]",3.453903,"[-38.13958090963712, 145.36252669303045]",3.840116,"[[-38.05104856543076, 145.3665303282505], [-38...",6.090210
1,50 Elmtree Crescent Clyde North VIC 3978,560.0,4,2,2.0,3978.0,"[-38.0825712, 145.3561984]",POINT (145.3561984 -38.0825712),212031555.0,Clyde North - North,...,"[-38.08468, 145.3638]",0.705427,"[-38.045325, 145.347181]",4.216162,"[-38.0604189, 145.3394612]",2.866025,"[-38.03401014838179, 145.37166338662394]",5.566924,"[[-38.05104856543076, 145.3665303282505], [-38...",3.619981


In [13]:
property_data['closest_station'][0]

['[-38.05104856543076, 145.3665303282505]',
 '[-38.0663192834778, 145.4116572812351]']

In [14]:
sample_property_data['closest_station'][0]

['[-38.05104856543076, 145.3665303282505]',
 '[-38.0663192834778, 145.4116572812351]']

# 3. Find the foot walk distance or duration
1. distance
2. duration
3. steps
4. way points

In [15]:
import requests

In [16]:
sample_property_data =  property_data.head(2)
sample_property_data

,name,rental_price,num_bedroom,num_bathroom,num_parking,postcode,coordinates,property_geometry,sa2_code,sa2_name,...,closest_school,school_distance(KM),closest_hospital,hospital_distance(KM),closest_mall,mall_distance(KM),closest_park,park_distance(KM),closest_station,station_distance(KM)
0,31 Chittagong Drive Clyde North VIC 3978,575.0,4,2,2.0,3978.0,"[-38.1053122, 145.3570863]",POINT (145.3570863 -38.1053122),212031556.0,Clyde North - South,...,"[-38.10602, 145.37876]",1.898006,"[-38.045325, 145.347181]",6.726397,"[-38.1184718, 145.3213262]",3.453903,"[-38.13958090963712, 145.36252669303045]",3.840116,"[[-38.05104856543076, 145.3665303282505], [-38...",6.090210
1,50 Elmtree Crescent Clyde North VIC 3978,560.0,4,2,2.0,3978.0,"[-38.0825712, 145.3561984]",POINT (145.3561984 -38.0825712),212031555.0,Clyde North - North,...,"[-38.08468, 145.3638]",0.705427,"[-38.045325, 145.347181]",4.216162,"[-38.0604189, 145.3394612]",2.866025,"[-38.03401014838179, 145.37166338662394]",5.566924,"[[-38.05104856543076, 145.3665303282505], [-38...",3.619981


In [17]:
property_data_0_900= property_data.iloc[0:900]
property_data_900_1800= property_data.iloc[900:1800]
property_data_1800_2700= property_data.iloc[1800:2700]
property_data_2700_3600= property_data.iloc[2700:3600]
property_data_3600_4500= property_data.iloc[3600:4500]
property_data_4500_5400= property_data.iloc[4500:5400]
property_data_5400_6300= property_data.iloc[5400:6300]
property_data_6300_7200= property_data.iloc[6300:7200]
property_data_7200_8100= property_data.iloc[7200:8100]
property_data_8100_8802= property_data.iloc[8100:8802]

In [18]:
def API_download(df,api_key):
    df['closest_distance_station(KM)'] = [[] for _ in range(len(df))]       
    df['duration_station(seconds)'] = [[] for _ in range(len(df))]
    df['way_out_station'] = [[] for _ in range(len(df))]
    
    for index, row in df.iterrows():
        coordinate_property_str = row['coordinates']
        coordinate_station_list = row['closest_station']
        
        # Check if the coordinate_property_str is a valid string
        if coordinate_property_str.startswith('[') and coordinate_property_str.endswith(']'):
            # Remove brackets and split by comma
            parts = coordinate_property_str.strip('[]').split(',')
            if len(parts) == 2:
                latitude_property = float(parts[0].strip())
                longitude_property = float(parts[1].strip())
                print(f"Property Coordinates: Longitude = {longitude_property}, Latitude = {latitude_property}")
        
        for i in range(len(coordinate_station_list)):
            coordinate_station_str = coordinate_station_list[i]
            # Check if the coordinate_station_str is a valid string
            if coordinate_station_str.startswith('[') and coordinate_station_str.endswith(']'):
                # Remove brackets and split by comma
                parts_station = coordinate_station_str.strip('[]').split(',')
                if len(parts_station) == 2:
                    latitude_station = float(parts_station[0].strip())
                    longitude_station = float(parts_station[1].strip())
                    
                    # Construct the API URL with start and end coordinates,# summer's key
                    api_web = f'https://api.openrouteservice.org/v2/directions/driving-car?api_key={api_key}&start={longitude_property},{latitude_property}&end={longitude_station},{latitude_station}'
                    print(api_web)
                    response = requests.get(api_web)
                    if response.status_code ==200:
                        d  = response.json()
                        d = response.json()
                        features = d.get('features', [])
                        summary = features[0]['properties']['summary']
                        df.at[index, 'closest_distance_station(KM)'].append(summary['distance'])
                        df.at[index, 'duration_station(seconds)'].append(summary['duration'])
                        df.at[index, 'way_out_station'].append(features[0]['properties']['way_points'])
                    else:
                        print(f'fail:{response.status_code}')
                        print(response.text)
    #sample_property_data.to_csv('../data/raw/external_data/open_route.csv',index =False)
    for index, row in df.iterrows():
        distance = row['closest_distance_station(KM)']
        duration = row['duration_station(seconds)']
        way_out = row['way_out_station']
        
        if distance:  # Check if the list is not empty
            minmum_distance = min(distance)
            minmum_duration = min(duration)
            way_out_chose = way_out[distance.index(minmum_distance)]  # Use the index to get the corresponding way_out
            
            df.at[index, 'closest_distance_station(KM)'] = minmum_distance
            #df.at[index, 'duration_station(seconds)'] = minmum_duration
            #df.at[index, 'way_out_station'] = way_out_chose
    
    return df   
                

In [19]:
property_data_0_900= property_data.iloc[0:900]
property_data_900_1800= property_data.iloc[900:1800]
property_data_1800_2700= property_data.iloc[1800:2700]
property_data_2700_3600= property_data.iloc[2700:3600]
property_data_3600_4500= property_data.iloc[3600:4500]
property_data_4500_5400= property_data.iloc[4500:5400]
property_data_5400_6300= property_data.iloc[5400:6300]
property_data_6300_7200= property_data.iloc[6300:7200]
property_data_7200_8100= property_data.iloc[7200:8100]
property_data_8100_8802= property_data.iloc[8100:8802]

In [20]:
#yin:5b3ce3597851110001cf6248e0a3d45b4797493ebb67f817d8b0520d
#wan:5b3ce3597851110001cf6248bae03afdf08048beb548d173480eaf09
#lei:5b3ce3597851110001cf62482cbd333f5aaa48f5937fc4777bb9689e
#tu:5b3ce3597851110001cf6248dc30edff982e4388a3c293540ed61878
#tu2: 5b3ce3597851110001cf6248d2d3206607ba4e7ea7be074a2f63f8b8
#wan2: 5b3ce3597851110001cf62486d8d552a2a9a4c3ea00a563023087ede
#dd:5b3ce3597851110001cf624854eeaa9dae3a4f8aaa3d49c75741b307
#dd1: 5b3ce3597851110001cf6248cdb2ab8025f04880a93df2d47b04127f
#tu3:5b3ce3597851110001cf624813fe2f12053c4c77b489ac5da2d123ed
#tu4: 5b3ce3597851110001cf62482f4f78906b5140be8bae8037f15b4bc3
#wan3: 5b3ce3597851110001cf6248d2be8d5a1800460981dab960d7c10c8f

In [ ]:
station_0_900= API_download(property_data_0_900,'5b3ce3597851110001cf6248e0a3d45b4797493ebb67f817d8b0520d')#yin
#station_0_900.to_csv('../data/raw/external_data/cloest_statoion_0_900.csv',index=False)


In [ ]:
# 1
station_0_900.to_csv('../data/raw/external_data/cloest_statoion_0_900.csv',index=False)

In [ ]:
# 2
station_900_1800= API_download(property_data_900_1800,'5b3ce3597851110001cf6248bae03afdf08048beb548d173480eaf09')#wan
station_900_1800.to_csv('../data/raw/external_data/cloest_statoion_900_1800.csv',index=False)


In [ ]:
# 3
station_1800_2700 = API_download(property_data_1800_2700,'5b3ce3597851110001cf62482cbd333f5aaa48f5937fc4777bb9689e')#lei
station_1800_2700.to_csv('../data/raw/external_data/cloest_statoion_1800_2700.csv',index=False)

In [ ]:
# 4
station_2700_3600= API_download(property_data_2700_3600,'5b3ce3597851110001cf6248dc30edff982e4388a3c293540ed61878')#tu
station_2700_3600.to_csv('../data/raw/external_data/cloest_statoion_2700_3600.csv',index=False)

In [ ]:
# 5
station_3600_4500= API_download(property_data_3600_4500,'5b3ce3597851110001cf624854eeaa9dae3a4f8aaa3d49c75741b307')#dd
station_3600_4500.to_csv('../data/raw/external_data/cloest_statoion_3600_4500.csv',index=False)

In [21]:
# 6
station_4500_5400= API_download(property_data_4500_5400,'5b3ce3597851110001cf6248d2d3206607ba4e7ea7be074a2f63f8b8')#tu2
station_4500_5400.to_csv('../data/raw/external_data/cloest_statoion_4500_5400.csv',index=False)

/tmp/ipykernel_20652/3044393849.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['closest_distance_station(KM)'] = [[] for _ in range(len(df))]
/tmp/ipykernel_20652/3044393849.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['duration_station(seconds)'] = [[] for _ in range(len(df))]
/tmp/ipykernel_20652/3044393849.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the docum

Property Coordinates: Longitude = 145.2038198, Latitude = -37.8226981
https://api.openrouteservice.org/v2/directions/driving-car?api_key=5b3ce3597851110001cf6248d2d3206607ba4e7ea7be074a2f63f8b8&start=145.2038198,-37.8226981&end=145.21453375482605,-37.818731170211926
https://api.openrouteservice.org/v2/directions/driving-car?api_key=5b3ce3597851110001cf6248d2d3206607ba4e7ea7be074a2f63f8b8&start=145.2038198,-37.8226981&end=145.1927343252876,-37.81806164158357
Property Coordinates: Longitude = 145.1971191, Latitude = -37.8194891
https://api.openrouteservice.org/v2/directions/driving-car?api_key=5b3ce3597851110001cf6248d2d3206607ba4e7ea7be074a2f63f8b8&start=145.1971191,-37.8194891&end=145.1927343252876,-37.81806164158357
https://api.openrouteservice.org/v2/directions/driving-car?api_key=5b3ce3597851110001cf6248d2d3206607ba4e7ea7be074a2f63f8b8&start=145.1971191,-37.8194891&end=145.21453375482605,-37.818731170211926
Property Coordinates: Longitude = 145.1940639, Latitude = -37.80713370000001

In [22]:
# 7
station_5400_6300= API_download(property_data_5400_6300,'5b3ce3597851110001cf62486d8d552a2a9a4c3ea00a563023087ede')#wan2
station_5400_6300.to_csv('../data/raw/external_data/cloest_statoion_5400_6300.csv',index=False)

/tmp/ipykernel_20652/3044393849.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['closest_distance_station(KM)'] = [[] for _ in range(len(df))]
/tmp/ipykernel_20652/3044393849.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['duration_station(seconds)'] = [[] for _ in range(len(df))]
/tmp/ipykernel_20652/3044393849.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the docum

Property Coordinates: Longitude = 144.7380713, Latitude = -37.8915404
https://api.openrouteservice.org/v2/directions/driving-car?api_key=5b3ce3597851110001cf62486d8d552a2a9a4c3ea00a563023087ede&start=144.7380713,-37.8915404&end=144.74578892239182,-37.87050531075657
https://api.openrouteservice.org/v2/directions/driving-car?api_key=5b3ce3597851110001cf62486d8d552a2a9a4c3ea00a563023087ede&start=144.7380713,-37.8915404&end=144.70164542039765,-37.88300835081249
Property Coordinates: Longitude = 144.719688, Latitude = -37.9002478
https://api.openrouteservice.org/v2/directions/driving-car?api_key=5b3ce3597851110001cf62486d8d552a2a9a4c3ea00a563023087ede&start=144.719688,-37.9002478&end=144.70164542039765,-37.88300835081249
https://api.openrouteservice.org/v2/directions/driving-car?api_key=5b3ce3597851110001cf62486d8d552a2a9a4c3ea00a563023087ede&start=144.719688,-37.9002478&end=144.74578892239182,-37.87050531075657
Property Coordinates: Longitude = 144.7707081, Latitude = -37.9138729
https://a

In [23]:
# 8
station_6300_7200= API_download(property_data_6300_7200,'5b3ce3597851110001cf6248cdb2ab8025f04880a93df2d47b04127f') #dd1
station_6300_7200.to_csv('../data/raw/external_data/cloest_statoion_6300_7200.csv',index=False)

/tmp/ipykernel_20652/3044393849.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['closest_distance_station(KM)'] = [[] for _ in range(len(df))]
/tmp/ipykernel_20652/3044393849.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['duration_station(seconds)'] = [[] for _ in range(len(df))]
/tmp/ipykernel_20652/3044393849.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the docum

Property Coordinates: Longitude = 145.0512022, Latitude = -37.8678836
https://api.openrouteservice.org/v2/directions/driving-car?api_key=5b3ce3597851110001cf6248cdb2ab8025f04880a93df2d47b04127f&start=145.0512022,-37.8678836&end=145.06296142086927,-37.86892689704469
https://api.openrouteservice.org/v2/directions/driving-car?api_key=5b3ce3597851110001cf6248cdb2ab8025f04880a93df2d47b04127f&start=145.0512022,-37.8678836&end=145.05821539372369,-37.859220165712905
Property Coordinates: Longitude = 145.0419122, Latitude = -37.8740808
https://api.openrouteservice.org/v2/directions/driving-car?api_key=5b3ce3597851110001cf6248cdb2ab8025f04880a93df2d47b04127f&start=145.0419122,-37.8740808&end=145.0424920017573,-37.87750503510752
https://api.openrouteservice.org/v2/directions/driving-car?api_key=5b3ce3597851110001cf6248cdb2ab8025f04880a93df2d47b04127f&start=145.0419122,-37.8740808&end=145.02938267628795,-37.86627535035504
Property Coordinates: Longitude = 145.0590084, Latitude = -37.8771406
https:

In [21]:
# 9 # use key:5b3ce3597851110001cf6248e0a3d45b4797493ebb67f817d8b0520d yin
station_7200_8100= API_download(property_data_7200_8100,'5b3ce3597851110001cf6248d2be8d5a1800460981dab960d7c10c8f') #wan3
station_7200_8100.to_csv('../data/raw/external_data/cloest_statoion_7200_8100.csv',index=False)

/tmp/ipykernel_5145/3044393849.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['closest_distance_station(KM)'] = [[] for _ in range(len(df))]
/tmp/ipykernel_5145/3044393849.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['duration_station(seconds)'] = [[] for _ in range(len(df))]
/tmp/ipykernel_5145/3044393849.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the document

Property Coordinates: Longitude = 144.9205978, Latitude = -37.7650549
https://api.openrouteservice.org/v2/directions/driving-car?api_key=5b3ce3597851110001cf6248d2be8d5a1800460981dab960d7c10c8f&start=144.9205978,-37.7650549&end=144.9191501689529,-37.76578332535559
https://api.openrouteservice.org/v2/directions/driving-car?api_key=5b3ce3597851110001cf6248d2be8d5a1800460981dab960d7c10c8f&start=144.9205978,-37.7650549&end=144.91605168156977,-37.756089051458915
Property Coordinates: Longitude = 144.9269132, Latitude = -37.7710638
https://api.openrouteservice.org/v2/directions/driving-car?api_key=5b3ce3597851110001cf6248d2be8d5a1800460981dab960d7c10c8f&start=144.9269132,-37.7710638&end=144.9218041482494,-37.77541636237498
https://api.openrouteservice.org/v2/directions/driving-car?api_key=5b3ce3597851110001cf6248d2be8d5a1800460981dab960d7c10c8f&start=144.9269132,-37.7710638&end=144.9191501689529,-37.76578332535559
Property Coordinates: Longitude = 144.9164246, Latitude = -37.7715326
https://

In [22]:
# 10
station_8100_8802= API_download(property_data_8100_8802, '5b3ce3597851110001cf62482f4f78906b5140be8bae8037f15b4bc3')#tu4
station_8100_8802.to_csv('../data/raw/external_data/cloest_statoion_8100_8801.csv',index=False)

/tmp/ipykernel_5145/3044393849.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['closest_distance_station(KM)'] = [[] for _ in range(len(df))]
/tmp/ipykernel_5145/3044393849.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['duration_station(seconds)'] = [[] for _ in range(len(df))]
/tmp/ipykernel_5145/3044393849.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the document

Property Coordinates: Longitude = 145.1805407, Latitude = -37.9657114
https://api.openrouteservice.org/v2/directions/driving-car?api_key=5b3ce3597851110001cf62482f4f78906b5140be8bae8037f15b4bc3&start=145.1805407,-37.9657114&end=145.17582454212,-37.96619708446432
https://api.openrouteservice.org/v2/directions/driving-car?api_key=5b3ce3597851110001cf62482f4f78906b5140be8bae8037f15b4bc3&start=145.1805407,-37.9657114&end=145.19097129813807,-37.9776280322612
Property Coordinates: Longitude = 145.1923953, Latitude = -37.9696494
https://api.openrouteservice.org/v2/directions/driving-car?api_key=5b3ce3597851110001cf62482f4f78906b5140be8bae8037f15b4bc3&start=145.1923953,-37.9696494&end=145.19097129813807,-37.9776280322612
https://api.openrouteservice.org/v2/directions/driving-car?api_key=5b3ce3597851110001cf62482f4f78906b5140be8bae8037f15b4bc3&start=145.1923953,-37.9696494&end=145.17582454212,-37.96619708446432
Property Coordinates: Longitude = 145.172432, Latitude = -37.9669016
https://api.ope

# 4. Combine all data sets together

In [24]:
import pandas as pd

# List of files
files = [
    "../data/raw/external_data/cloest_statoion_0_900.csv",
    "../data/raw/external_data/cloest_statoion_900_1800.csv",
    "../data/raw/external_data/cloest_statoion_1800_2700.csv",
    "../data/raw/external_data/cloest_statoion_2700_3600.csv",
    "../data/raw/external_data/cloest_statoion_3600_4500.csv",
    "../data/raw/external_data/cloest_statoion_4500_5400.csv",
    "../data/raw/external_data/cloest_statoion_5400_6300.csv",
    "../data/raw/external_data/cloest_statoion_6300_7200.csv",
    "../data/raw/external_data/cloest_statoion_7200_8100.csv",
     "../data/raw/external_data/cloest_statoion_8100_8801.csv"
]

# Load and concatenate all files into one dataframe
df_combined = pd.concat([pd.read_csv(file) for file in files], ignore_index=True)
df_combined.shape

(8802, 29)

In [25]:
# Convert the 'closest_distance_station(KM)' column to float, handling NA values
df_combined['closest_distance_station(KM)'] = pd.to_numeric(df_combined['closest_distance_station(KM)'], errors='coerce')

# Perform the division
df_combined['closest_distance_station(KM)'] /= 1000

df_combined[['name', 'closest_distance_station(KM)']].head()


# Delete the 'way out station' and 'duration' columns, if they exist
columns_to_drop = ['duration_station(seconds)', 'way_out_station']
for col in columns_to_drop:
    if col in df_combined.columns:
        df_combined.drop(col, axis=1, inplace=True)

# Save the resulting dataframe to a new CSV file
output_path = "../data/curated/external_data/combined_cloest_station.csv"
df_combined.to_csv(output_path, index=False)

output_path

'../data/curated/external_data/combined_cloest_station.csv'

In [26]:
# Read the processed data
df_processed = pd.read_csv('../data/curated/external_data/combined_cloest_station.csv')
df_processed

,name,rental_price,num_bedroom,num_bathroom,num_parking,postcode,coordinates,property_geometry,sa2_code,sa2_name,...,school_distance(KM),closest_hospital,hospital_distance(KM),closest_mall,mall_distance(KM),closest_park,park_distance(KM),closest_station,station_distance(KM),closest_distance_station(KM)
0,31 Chittagong Drive Clyde North VIC 3978,575.0,4,2,2.0,3978.0,"[-38.1053122, 145.3570863]",POINT (145.3570863 -38.1053122),212031556.0,Clyde North - South,...,1.898006,"[-38.045325, 145.347181]",6.726397,"[-38.1184718, 145.3213262]",3.453903,"[-38.13958090963712, 145.36252669303045]",3.840116,"['[-38.05104856543076, 145.3665303282505]', '[...",6.090210,11.8866
1,50 Elmtree Crescent Clyde North VIC 3978,560.0,4,2,2.0,3978.0,"[-38.0825712, 145.3561984]",POINT (145.3561984 -38.0825712),212031555.0,Clyde North - North,...,0.705427,"[-38.045325, 145.347181]",4.216162,"[-38.0604189, 145.3394612]",2.866025,"[-38.03401014838179, 145.37166338662394]",5.566924,"['[-38.05104856543076, 145.3665303282505]', '[...",3.619981,5.3735
2,7 Mortdale Lane Clyde North VIC 3978,490.0,2,2,1.0,3978.0,"[-38.0961758, 145.3800644]",POINT (145.3800644 -38.0961758),212031556.0,Clyde North - South,...,1.100561,"[-38.045325, 145.347181]",6.344909,"[-38.0604189, 145.3394612]",5.332842,"[-38.13958090963712, 145.36252669303045]",5.064419,"['[-38.0663192834778, 145.4116572812351]', '[-...",4.320648,8.9170
3,54 Walhallow Drive Clyde North VIC 3978,540.0,4,2,1.0,3978.0,"[-38.1133324, 145.3457396]",POINT (145.3457396 -38.1133324),212031556.0,Clyde North - South,...,0.674921,"[-38.113312, 145.280832]",5.678594,"[-38.1184718, 145.3213262]",2.210922,"[-38.13958090963712, 145.36252669303045]",3.267265,"['[-38.0995872148084, 145.2804738077291]', '[-...",5.911467,8.1066
4,10 Sicily Road Clyde North VIC 3978,520.0,4,2,2.0,3978.0,"[-38.1295789, 145.3642993]",POINT (145.3642993 -38.1295789),212031303.0,Cranbourne South,...,2.225124,"[-38.113312, 145.280832]",7.522231,"[-38.1184718, 145.3213262]",3.956745,"[-38.13958090963712, 145.36252669303045]",1.122928,"['[-38.0995872148084, 145.2804738077291]', '[-...",8.056215,10.1328
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8797,61 Tongue Street Yarraville VIC 3013,630.0,2,1,0.0,3013.0,"[-37.8131463, 144.8909053]",POINT (144.8909053 -37.8131463),213031352.0,Yarraville,...,0.107655,"[-37.800508, 144.895155]",1.454065,"[-37.8016476, 144.8977057]",1.411290,"[-37.83084513979606, 144.91345002848863]",2.791844,"['[-37.81567190405462, 144.89006145414797]', '...",0.290453,1.1480
8798,47 Mill Avenue Yarraville VIC 3013,730.0,4,3,2.0,3013.0,"[-37.8222822, 144.872198]",POINT (144.872198 -37.8222822),213031352.0,Yarraville,...,0.239821,"[-37.797407, 144.887421]",3.072331,"[-37.828989, 144.84627]",2.396280,"[-37.83084513979606, 144.91345002848863]",3.746179,"['[-37.83072411396968, 144.88584876958876]', '...",1.522703,2.1264
8799,12 Adeney Street Yarraville VIC 3013,450.0,3,1,2.0,3013.0,"[-37.8163817, 144.8666543]",POINT (144.8666543 -37.8163817),213031352.0,Yarraville,...,0.819385,"[-37.797407, 144.887421]",2.789294,"[-37.828989, 144.84627]",2.273966,"[-37.83084513979606, 144.91345002848863]",4.413664,"['[-37.7993395010312, 144.86344949623717]', '[...",1.915810,2.8147
8800,229B Somerville Road Yarraville VIC 3013,300.0,1,1,0.0,3013.0,"[-37.8124289, 144.8779569]",POINT (144.8779569 -37.8124289),213031352.0,Yarraville,...,0.290245,"[-37.797407, 144.887421]",1.865866,"[-37.8016476, 144.8977057]",2.108881,"[-37.83084513979606, 144.91345002848863]",3.729966,"['[-37.81567190405462, 144.89006145414797]', '...",1.122803,1.8488
